# Linref Usage Guide

This guide covers the essential features and usage patterns for linref v1.0.

## Installation

Linref can easily be installed using `pip`. The package is dependent on a few major libraries (`geopandas`, `numpy`, `scipy`) which may require additional effort on some machines. Please review documentation for those packages as needed if any issues arise.

```
pip install linref
```

## Core Functionality

First, import `linref` with the shortened `lr` for ease of access and consistency with the dataframe accessor.

In [1]:
import linref as lr

### Load Sample Datasets

In [2]:
# Load sample roadway, crash, and pavement datasets
# By default, LRS is not configured - you must set it up
roadways = lr.datasets.load('roadways')
crashes  = lr.datasets.load('crashes')
pavement = lr.datasets.load('pavement')

# Or load with LRS pre-configured using set_lrs=True
# roadways = lr.datasets.load('roadways', set_lrs=True)

print(roadways.head(3))

    route  beg  end  traffic_volume  speed_limit                     geometry
0  US-101  0.0  2.5           15000           45    LINESTRING (0 0, 2.5 0.5)
1  US-101  2.5  5.0           22000           55  LINESTRING (2.5 0.5, 5 1.2)
2  US-101  5.0  7.8           18500           55    LINESTRING (5 1.2, 7.8 2)


### Setting Up an Existing LRS

**Method 1: Set LRS per DataFrame**

In [3]:
roadways = roadways.lr.set_lrs(
    key_col=['route'],
    loc_col='loc', # Should be defined even if not present in all datasets
    beg_col='beg',
    end_col='end',
    geom_col='geometry',
    geom_m_col='geometry_m', # Not currently present but will be generated later
    closed='left_mod' # 'left_mod' is a the recommended convention for most linear datasets
)
print(roadways.lr)

LRS_Accessor with linear referencing system (LRS):
[GR lc LN SP sm] LRS(key_col=['route'], loc_col='loc', beg_col='beg', end_col='end', geom_col='geometry', geom_m_col='geometry_m', closed='left_mod')


**Method 2: Set Default LRS**

In [4]:
# Define default LRS once
default_lrs = lr.LRS(
    key_col=['route'],
    loc_col='loc', # Should be defined even if not present in all datasets
    beg_col='beg',
    end_col='end',
    geom_col='geometry',
    geom_m_col='geometry_m', # Not currently present but will be generated later
    closed='left_mod' # 'left_mod' is a the recommended convention for most linear datasets
)

# Set as default for all DataFrames
lr.set_default_lrs(default_lrs)

# Now all DataFrames will use this LRS unless otherwise specified
print(crashes.lr)

LRS_Accessor with linear referencing system (LRS):
[GR LC ln SP sm] LRS(key_col=['route'], loc_col='loc', beg_col='beg', end_col='end', geom_col='geometry', geom_m_col='geometry_m', closed='left_mod')


## Essential Operations

### Selecting Events by Group

If you simply need to access or analyze the dataframe, grouped by the unique groups represented in the LRS's key column(s), you can do that with the `get_group` or `iter_groups` methods.

In [5]:
# Get events for a specific group
print(crashes.lr.get_group('SR-1')[['crash_id', 'route', 'loc']])

# Iterate over all groups to subset the data by unique groups
for group_name, group_df in crashes.lr.iter_groups():
    print(f"Group {group_name} has {len(group_df)} events")

# You can similarly get counts of events per group with the group_counts method
print(crashes.lr.group_counts())

    crash_id route   loc
6          7  SR-1  2.59
7          8  SR-1  7.62
8          9  SR-1  3.54
9         10  SR-1  6.00
11        12  SR-1  6.76
16        17  SR-1  3.74
17        18  SR-1  2.25
Group ('I-5',) has 4 events
Group ('SR-1',) has 7 events
Group ('US-101',) has 9 events
(I-5,)       4
(SR-1,)      7
(US-101,)    9
dtype: int64


### Dissolving Events

Merge consecutive linear events within the same group based on shared event begin and end points. To dissolve by additional attributes, use the `retain` parameter which accepts a list of one or more dataframe column labels.

In [6]:
# Dissolve events, keeping specific columns
dissolved = roadways.lr.dissolve()

# Note that the dissolved_index column links back to the original events
view_columns = ['route', 'beg', 'end', 'dissolved_index']
print(dissolved[view_columns])

# Use the retain parameter to keep additional columns
print(roadways.lr.dissolve(retain=['speed_limit'])[view_columns + ['speed_limit']])

    route  beg   end dissolved_index
0     I-5  0.0  12.0       [7, 8, 9]
1    SR-1  0.0   9.8       [4, 5, 6]
2  US-101  0.0  10.5    [0, 1, 2, 3]
    route  beg   end dissolved_index  speed_limit
0     I-5  0.0  12.0       [7, 8, 9]           65
1    SR-1  0.0   3.2             [4]           35
2    SR-1  3.2   6.5             [5]           45
3    SR-1  6.5   9.8             [6]           55
4  US-101  0.0   2.5             [0]           45
5  US-101  7.8  10.5             [3]           45
6  US-101  2.5   7.8          [1, 2]           55


The dissolve function will also merge geometries by default, retaining event bound information by upgrading geometries to be m-enabled.

In [7]:
# View the geometries created by the dissolve operation
print(dissolved.iloc[0].beg, dissolved.iloc[0].end)
print(dissolved.iloc[0].geometry)
print(dissolved.iloc[0].geometry_m)

0.0 12.0
LINESTRING (0 10, 4.1 11, 8.3 12.5, 12 14.2)
LINESTRING M (0.0 10.0 0.0, 4.1 11.0 4.1, 8.3 12.5 8.3, 12.0 14.2 12.0)


NOTE: The LINESTRING M shown here is using linref's `LineStringM` class located in the events.geometry module. This is an implementation of m-enabled linear geometries that provides an extension of the existing `shapely.LineString` with the features needed for linref's core functionality. Future versions of shapely are expected to provide native support for m-enabled linear geometries at which point linref will transition to their implementation. For now, please be cautious and review appropriate documentation if you are intending to use exports of these geometries in other programs. Use the WKT features of the `LineStringM` class and the linref `LRS_Accessor` class to support interoperability for now.

### Resegmentation

Continuing with data engineering, we can take the dissolved roadways layer and resegment it to a fixed length using the `resegment` method:

In [8]:
# Create 5-mile segments
segmented = dissolved.lr.resegment(length=5)
print(segmented.lr.get_group('I-5')[['route', 'beg', 'end']])

  route   beg   end
0   I-5   0.0   5.0
1   I-5   5.0  10.0
2   I-5  10.0  12.0


Note that by default, the final segment is allowed to be shorter than others if the length of the parent route isn't a multiple of the desired length. This behavior can be modified using the `fill` parameter:

In [16]:
# Various fill parameter options
resegmented = dissolved.lr.resegment(length=5, fill='cut') # cut (default)
print(resegmented.lr.get_group('I-5')[['route', 'beg', 'end']])

resegmented = dissolved.lr.resegment(length=5, fill='extend') # extend
print(resegmented.lr.get_group('I-5')[['route', 'beg', 'end']])

resegmented = dissolved.lr.resegment(length=5, fill='balance') # balance (cut when long, extend when short)
print(resegmented.lr.get_group('I-5')[['route', 'beg', 'end']])
# Not shown: right, left

  route   beg   end
0   I-5   0.0   5.0
1   I-5   5.0  10.0
2   I-5  10.0  12.0
  route  beg   end
0   I-5  0.0   5.0
1   I-5  5.0  12.0
  route  beg   end
0   I-5  0.0   5.0
1   I-5  5.0  12.0


### Relating Datasets

An important functionality of linearly referenced data is events-based conflation. Linref makes this easy with the `relate` method, which builds dynamic relationships between two linearly referenced dataframes that can then be aggregated with a variety of methods.

#### Conflating *point* data onto *linear* data

In [17]:
# Conflating point data onto linear data
#          left_df               right_df   <-- relations use terminology of left and right
relation = resegmented.lr.relate(crashes)

# Dataframe-level aggregations like count are easy
print(relation.count()) # Count the number of crashes on each roadway segment

# Aggregator results are designed to be assigned as new columns to the left dataframe
resegmented['crash_counts'] = relation.count()
print(resegmented.head(3)[['route', 'beg', 'end', 'crash_counts']])

[1 3 4 3 5 4]
  route  beg   end  crash_counts
0   I-5  0.0   5.0             1
1   I-5  5.0  12.0             3
2  SR-1  0.0   5.0             4


Column-level aggregators can be accessed using column indexing on the `relation` instance, using a syntax similar to the pandas `groupby` syntax:

In [26]:
# List of unique values (creates a single column)
resegmented['crash_ids'] = relation['crash_id'].list()
# Apply an aggregator to multiple columns at once
resegmented[['crash_ids', 'severities']] = relation[['crash_id', 'severity']].list()
# Counts of unique values (creates a number of columns equal to the number of unique values)
severities = ['Fatal', 'Injury', 'Property Damage Only']
resegmented[severities] = relation['severity'].value_counts()[severities] # Returned as a dataframe

print(resegmented.head(3)[['route', 'beg', 'end', 'crash_ids', 'severities'] + severities])

  route  beg   end       crash_ids  \
0   I-5  0.0   5.0             [3]   
1   I-5  5.0  12.0      [5, 6, 13]   
2  SR-1  0.0   5.0  [7, 9, 17, 18]   

                                      severities  Fatal  Injury  \
0                                       [Injury]    0.0       1   
1                        [Fatal, Injury, Injury]    1.0       2   
2  [Fatal, Injury, Injury, Property Damage Only]    1.0       2   

   Property Damage Only  
0                   0.0  
1                   0.0  
2                   1.0  


#### Conflating *linear* data onto *linear* data

In [35]:
# Conflating linear data onto linear data
relation = resegmented.lr.relate(pavement)

# Length-weighted mean of numerical data
resegmented['condition_rating'] = relation['condition_rating'].mean().round(3)
resegmented['condition_ratings'] = relation['condition_rating'].list()

print(resegmented.head(3)[['route', 'beg', 'end', 'condition_rating', 'condition_ratings']])

# Length-weighted mode of categorical data
resegmented['surface_type'] = relation['surface_type'].mode()
resegmented['surface_types'] = relation['surface_type'].list()

print(resegmented.head(3)[['route', 'beg', 'end', 'surface_type', 'surface_types']])

  route  beg   end  condition_rating condition_ratings
0   I-5  0.0   5.0            81.900          [84, 77]
1   I-5  5.0  12.0            81.278      [77, 83, 82]
2  SR-1  0.0   5.0            78.571      [79, 73, 88]
  route  beg   end surface_type                  surface_types
0   I-5  0.0   5.0      Asphalt            [Asphalt, Concrete]
1   I-5  5.0  12.0     Concrete  [Concrete, Asphalt, Concrete]
2  SR-1  0.0   5.0     Concrete  [Concrete, Concrete, Asphalt]


#### Creating new geometries from relations

We can create new geometries for linearly referenced point or linear dataframes by relating them to linearly referenced dataframes containing m-enabled geometries. When dissolving the roadways layer earlier, that method automatically created m-enabled geometries to retain dissolved event information. We can create them manually for other layers using the `add_geom_m` method, which applies event boundaries to the existing geometries.

In [56]:
# Add m-enabled geometries to the roadways layer
roadways = roadways.lr.add_geom_m()

# Interpolate new crash geometries based on their LRS information
interpolated = crashes.lr.relate(roadways).interpolate()
print(list(crashes.geometry)[:3])
print(list(interpolated)[:3])

# Cut new roadway geometries from the parent LRS layer
cut = roadways.lr.relate(dissolved).cut()
print(list(roadways.geometry_m)[:3])
print(list(cut)[:3])


[<POINT (2.56 0.52)>, <POINT (8.18 2.15)>, <POINT (4.47 11.13)>]
[<POINT (2.56 0.517)>, <POINT (8.18 2.155)>, <POINT (4.47 11.132)>]
[LINESTRING M (0.0 0.0 0.0, 2.5 0.5 2.5) # linref compatibility approximation, LINESTRING M (2.5 0.5 2.5, 5.0 1.2 5.0) # linref compatibility approximation, LINESTRING M (5.0 1.2 5.0, 7.8 2.0 7.8) # linref compatibility approximation]
[LINESTRING M (0.0 0.0 0.0, 2.5 0.5 2.5) # linref compatibility approximation, LINESTRING M (2.5 0.5 2.5, 5.0 1.2 5.0) # linref compatibility approximation, LINESTRING M (5.0 1.2 5.0, 7.8 2.0 7.8) # linref compatibility approximation]


## Working with Point Events

### Point-Based Data

In [ ]:
# Point events with location column
signs = pd.DataFrame({
    'route': ['A', 'A', 'B'],
    'milepost': [0.5, 1.2, 0.8],
    'sign_type': ['Speed Limit', 'Stop', 'Yield']
}).lr.set_lrs(
    key_col=['route'],
    loc_col='milepost'
)

### Converting Point to Linear Events

In [ ]:
# Extend point events to create buffer zones
sign_zones = signs.lr.extend(
    extend_begs=0.1,  # 0.1 before
    extend_ends=0.1   # 0.1 after
)

## Geometry Operations

### Generating Linear Events from Geometries

If you have a GeoDataFrame with route geometries but no mile markers:

In [ ]:
import geopandas as gpd

# GeoDataFrame with route geometries
routes_gdf = gpd.read_file('routes.geojson')

routes = routes_gdf.lr.set_lrs(
    key_col=['route'],
    geom_col='geometry'
)

# Generate begin/end mile markers from geometry lengths
routes = routes.lr.generate_linear_events(
    beg_col='beg_mp',
    end_col='end_mp',
    scale=1/5280,  # Convert feet to miles
    decimals=3,
    add_geom_m=True
)

### Cutting Geometries

Cut geometries for your events from reference routes:

In [ ]:
# Events with mile markers, need geometries
events_df = events_df.lr.set_lrs(
    key_col=['route'],
    beg_col='beg',
    end_col='end'
)

# Reference routes with M-enabled geometries
ref_routes = ref_gdf.lr.set_lrs(
    key_col=['route'],
    geom_m_col='geometry_m'
)

# Cut geometries for events
events_gdf = events_df.lr.cut_from(ref_routes)

### Extracting M-Values

Extract begin and end M-values from M-enabled geometries:

In [ ]:
df = df.lr.extract_m_values(
    beg_col='extracted_beg',
    end_col='extracted_end'
)

## Integrating Multiple Datasets

Combine multiple event datasets into a unified framework:

In [ ]:
# Multiple datasets with matching LRS
pavement_df = pavement_df.lr.set_lrs(
    key_col=['route'],
    beg_col='beg',
    end_col='end'
)
traffic_df = traffic_df.lr.set_lrs(
    key_col=['route'],
    beg_col='beg',
    end_col='end'
)
crash_df = crash_df.lr.set_lrs(
    key_col=['route'],
    beg_col='beg',
    end_col='end'
)

# Integrate all datasets at once
integrated_df = pavement_df.lr.integrate(
    [traffic_df, crash_df],
    fill_gaps=False,
    split_at_locs=False
)

## Common Patterns

### Pattern 1: Dissolving by Attributes

In [ ]:
# Dissolve consecutive events with same attributes
dissolved = df.lr.dissolve(
    retain=['speed_limit', 'pavement_type'],
    inverse_index=True,  # Track original events
    inverse_col='original_ids'
)

### Pattern 2: Crash Analysis

In [ ]:
# Count and analyze crashes on road segments
relation = roads.lr.relate(crashes)

# Count crashes per segment
roads['crash_count'] = relation.count()

# Get crash severity data
crash_severity = crashes['severity'].values
roads['avg_severity'] = relation.mean(data=crash_severity)

### Pattern 3: Uniform Segmentation

In [ ]:
# Create uniform 0.5-mile segments
segmented = roads.lr.resegment(length=0.5)

### Pattern 4: Buffering Point Events

In [ ]:
# Convert point events to linear with buffers
signs = signs_df.lr.set_lrs(
    key_col=['route'],
    loc_col='milepost'
)

# Extend to create buffer zones
sign_zones = signs.lr.extend(
    extend_begs=0.25,  # 0.25 miles before
    extend_ends=0.25   # 0.25 miles after
)

## Performance Tips

### 1. Sort Before Processing

In [ ]:
# Sort events before dissolve or other operations
df = df.lr.sort_standard()

### 2. Drop Invalid Events

In [ ]:
# Remove events with missing data
df_clean = df.lr.drop_invalid_events()

### 3. Use Chunking for Large Overlays

In [ ]:
# Control memory usage with chunking
overlay_matrix = df1.lr.overlay(df2, chunksize=500)

### 4. Cache Relations

In [ ]:
# Enable caching (default) for multiple operations
relation = df1.lr.relate(df2, cache=True)

# Multiple operations reuse cached data
counts = relation.count()
first_vals = relation.first()

## API Reference Summary

### DataFrame Methods (via `.lr` accessor)

- **Configuration**: `set_lrs()`, `modify_lrs()`, `clear_lrs()`
- **Selection**: `get_group()`, `iter_groups()`
- **Transformation**: `dissolve()`, `resegment()`, `extend()`, `shift()`, `round()`
- **Relationships**: `relate()`, `overlay()`, `integrate()`
- **Geometry**: `generate_linear_events()`, `add_geom_m()`, `extract_m_values()`, `cut_from()`, `interpolate_from()`
- **Validation**: `drop_invalid_events()`, `set_monotonic()`

### EventsRelation Methods

- **Aggregation**: `count()`, `first()`, `last()`, `sum()`, `mean()`
- **Geometry**: `cut()`, `interpolate()`
- **Data Transfer**: `distribute()`

## Troubleshooting

### "No LRS set"

Make sure you've called `.set_lrs()` on your DataFrame:

In [ ]:
df = df.lr.set_lrs(key_col=['route'], beg_col='beg', end_col='end')

### "LRS columns not found"

Verify that the column names in your LRS match your DataFrame:

In [ ]:
print(df.columns)
print(df.lr.lrs)  # Check current LRS configuration

### Missing Groups in Relations

Relate and overlay operations require matching groups:

In [ ]:
# Check LRS configuration
print("Keys in df1:", df1.lr.key_col)
print("Keys in df2:", df2.lr.key_col)

# Iterate groups to verify data
for group, group_df in df1.lr.iter_groups():
    print(f"Group {group}: {len(group_df)} events")

## Additional Resources

- **API Documentation**: https://linref.readthedocs.io/
- **GitHub**: https://github.com/tariqshihadah/linref
- **Issues**: https://github.com/tariqshihadah/linref/issues